## IMPORT LIBRARY

In [1]:
import os
import sys
import time
import pyarrow as pa  # Opsional: Hanya untuk pamer/cek versi
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from xgboost.spark import SparkXGBClassifier 
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import pyspark.sql.functions as F
import pandas as pd
import numpy as np

## 1. INISIALISASI SPARK

In [2]:
# Bersihkan sesi lama jika masih nyangkut
if 'spark' in locals():
    try: spark.stop()
    except: pass

# Gunakan nama file JAR yang WAJIB dipakai (sesuai arahan awal)
path_jar = "clickhouse-jdbc-0.6.3.jar"

# Pastikan file benar-benar ada sebelum Spark dijalankan
if not os.path.exists(path_jar):
    print(f"❌ AWAS: File JAR tidak ditemukan di '{path_jar}'!")
    print(f"Pastikan file {path_jar} ada di folder proyekmu.")
else:
    print("✅ File JAR ditemukan! Memulai SparkSession dalam Mode Maximum Memory (10GB)...")

    python_path = sys.executable
    os.environ["PYSPARK_PYTHON"] = python_path
    os.environ["PYSPARK_DRIVER_PYTHON"] = python_path
    os.environ["HADOOP_HOME"] = "D:/Pyta/hadoop"
    os.environ["PATH"] = os.environ["PATH"] + ";D:/Pyta/hadoop/bin"
    
    # Kita set Heap Memory 8GB + Off-Heap 2GB = Total 10GB RAM
    spark = SparkSession.builder \
        .appName("FlightDelayModelBuilder") \
        .config("spark.jars", path_jar) \
        .config("spark.driver.extraClassPath", path_jar) \
        .config("spark.executor.extraClassPath", path_jar) \
        .config("spark.executor.memory", "9g") \
        .config("spark.driver.memory", "9g") \
        .config("spark.driver.maxResultSize", "3g") \
        .config("spark.memory.offHeap.enabled", "true") \
        .config("spark.memory.offHeap.size", "2g") \
        .config("spark.pyspark.python", python_path) \
        .config("spark.pyspark.driver.python", python_path) \
        .config("spark.executorEnv.PYSPARK_PYTHON", python_path) \
        .config("spark.executorEnv.PYSPARK_DRIVER_PYTHON", python_path) \
        .getOrCreate()

    print("✅ SparkSession lokal (Mode 10GB) berhasil terhubung!")

✅ File JAR ditemukan! Memulai SparkSession dalam Mode Maximum Memory (10GB)...
✅ SparkSession lokal (Mode 10GB) berhasil terhubung!


## 2. MEMBACA DATA DARI AWS

In [3]:
# CLICKHOUSE_URL = "jdbc:clickhouse://13.215.79.3:8123/flight_delay?compress=false&connection_timeout=120000&socket_timeout=120000&http_connection_provider=HTTP_URL_CONNECTION"

# print("📥 Menarik data dari AWS dengan 31 Partisi (Berdasarkan Tanggal)...")

# # Kita ganti partitionColumn menjadi DayofMonth (tanggal 1 s/d 31)
# df_all = spark.read.format("jdbc") \
#     .option("url", CLICKHOUSE_URL) \
#     .option("dbtable", "ontime_features") \
#     .option("user", "default") \
#     .option("password", "rahasia123") \
#     .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
#     .option("partitionColumn", "flight_month") \
#     .option("lowerBound", "1") \
#     .option("upperBound", "12") \
#     .option("numPartitions", "12") \
#     .option("fetchsize", "10000") \
#     .load()

# print("✅ Skema data berhasil dimuat dengan 31 partisi mikro!")

In [4]:
import time

print("⚡ [BYPASS AWS] Membaca data langsung dari file lokal...")
start_time = time.time()

# ---------------------------------------------------------
# TAHAP 1: MUAT DATA LOKAL
# (Sesuaikan dengan format file yang dikirim Putri)
# ---------------------------------------------------------

# JIKA PUTRI MENGIRIM FILE PARQUET (Sangat disarankan & lebih cepat):
df_all = spark.read.parquet("../data/ontime_features.parquet")

# JIKA PUTRI MENGIRIM FILE CSV (Hapus tanda '#' di bawah ini jika pakai CSV):
# df_all = spark.read.csv("../data/ontime_features.csv", header=True, inferSchema=True)

⚡ [BYPASS AWS] Membaca data langsung dari file lokal...


In [5]:
# Tambahkan cell baru di awal, setelah data dimuat
# Lihat SEMUA kolom yang ada di dataset
print("Semua kolom di ontime_features:")
for col_name, dtype in df_all.dtypes:
    print(f"  {col_name}: {dtype}")

Semua kolom di ontime_features:
  FlightDate: string
  IATA_CODE_Reporting_Airline: string
  Flight_Number_Reporting_Airline: bigint
  Origin: string
  Dest: string
  CRSDepTime: bigint
  flight_year: bigint
  flight_quarter: bigint
  flight_month: bigint
  flight_day: bigint
  day_of_week: bigint
  is_weekend: bigint
  season: string
  dep_hour: bigint
  arr_hour: bigint
  dep_time_bucket: string
  arr_time_bucket: string
  route: string
  same_state_route: bigint
  distance_bucket: bigint
  CRSElapsedTime: bigint
  Distance: bigint
  DistanceGroup: bigint
  OriginAirportID: bigint
  OriginState: string
  DestAirportID: bigint
  DestState: string
  route_avg_arr_delay_prev: string
  route_arr_delay_rate_prev: double
  route_cancel_rate_prev: double
  carrier_arr_delay_rate_prev: double
  carrier_cancel_rate_prev: double
  origin_arr_delay_rate_prev: double
  origin_cancel_rate_prev: double
  dest_arr_delay_rate_prev: double
  dest_cancel_rate_prev: double
  route_carrier_arr_delay_rat

In [6]:
# Cek keberadaan kolom-kolom kritis
kolom_kritis = [
    "Origin", "Dest",           # airport kode IATA
    "Reporting_Airline",        # kode maskapai
    "Tail_Number",              # nomor ekor pesawat
    "DepDelay", "DepDel15",     # delay keberangkatan
    "TaxiOut",                  # waktu taxi
    "Distance",                 # jarak dalam miles (bukan bucket)
    "CRSDepTime", "CRSArrTime", # jadwal waktu
    "DayOfWeek",                # hari dalam seminggu (1-7)
]

kolom_tersedia = [c for c, _ in df_all.dtypes]
print("\nKolom kritis yang TERSEDIA:")
for k in kolom_kritis:
    status = "✅" if k in kolom_tersedia else "❌"
    print(f"  {status} {k}")


Kolom kritis yang TERSEDIA:
  ✅ Origin
  ✅ Dest
  ❌ Reporting_Airline
  ❌ Tail_Number
  ❌ DepDelay
  ❌ DepDel15
  ❌ TaxiOut
  ✅ Distance
  ✅ CRSDepTime
  ❌ CRSArrTime
  ❌ DayOfWeek


## 3. TIME-BASED SPLIT

In [7]:
# import time

# print("📅 [STRATEGI INTEGRASI] Memulai Pemisahan Waktu & Unduh AWS (12 Partisi Bulan)...")

# # Definisikan pemisahan dasar langsung ke variabel target
# df_train_raw = df_all.filter(df_all.FlightDate < "2025-01-01")
# df_test_raw = df_all.filter(df_all.FlightDate >= "2025-01-01")

# # ---------------------------------------------------------
# # TAHAP A: TRAINING DATA (Pre-trigger & Kunci di RAM)
# # ---------------------------------------------------------
# print("\n📥 Tahap A: Mengunduh Data Training (2021-2024) ke RAM...")
# start_train = time.time()

# # Pre-trigger per BULAN (1-12) agar sejalan dengan Cell 2
# for month in range(1, 13):
#     print(f"   ⏳ [Partisi {month}/12] Menyedot data penerbangan Bulan ke-{month:02d} dari AWS...")
#     # Ubah filter menggunakan 'flight_month'
#     df_train_raw.filter(df_train_raw.flight_month == month).cache().count()

# # Kunci keseluruhan data training di RAM
# df_train_raw = df_train_raw.cache()
# train_count = df_train_raw.count()
# print(f"   ✔️ Sukses! {train_count:,} baris Data Training terkunci di RAM. Durasi: {time.time() - start_train:.2f} detik.")

# # ---------------------------------------------------------
# # TAHAP B: TESTING DATA (Pre-trigger & Kunci di RAM)
# # ---------------------------------------------------------
# print("\n📥 Tahap B: Mengunduh Data Testing (Tahun 2025) ke RAM...")
# start_test = time.time()

# # Kunci keseluruhan data testing di RAM
# df_test_raw = df_test_raw.cache()
# test_count = df_test_raw.count()
# print(f"   ✔️ Sukses! {test_count:,} baris Data Testing terkunci di RAM. Durasi: {time.time() - start_test:.2f} detik.")

# # ---------------------------------------------------------
# print("\n✅ Selesai! Pipa JDBC AWS ditutup. Komputasi ML selanjutnya 100% menggunakan RAM lokal!")

# ---------------------------------------------------------
# TAHAP 2: PEMISAHAN WAKTU & KUNCI DI RAM
# ---------------------------------------------------------
print("⏳ Memisahkan data Training (2021-2024) dan Testing (2025)...")

# OPSI 1: Data 2023-2024 dapat bobot penuh, 2021-2022 dapat bobot berkurang
df_train_raw = df_all.filter(df_all.FlightDate < "2025-01-01").cache()
df_test_raw  = df_all.filter(df_all.FlightDate >= "2025-01-01").cache()

# # OPSI 2 : Hanya menggunakan data pada tahun 2023-2024 untuk training (bobot penuh), dan 2025 untuk testing
# df_train_raw = df_all.filter(
#     (df_all.FlightDate >= "2023-01-01") & (df_all.FlightDate < "2025-01-01")
# ).cache()

# Memicu Spark untuk benar-benar menarik data ke RAM sekarang juga
train_count = df_train_raw.count()
test_count = df_test_raw.count()

print(f"\n✔️ Sukses memuat data ke RAM! Waktu eksekusi: {time.time() - start_time:.2f} detik.")
print(f"📊 Jumlah Data Training : {train_count:,} baris")
print(f"📊 Jumlah Data Testing  : {test_count:,} baris")
print("✅ Selesai! Komputasi ML selanjutnya berjalan 100% Offline & Kilat!")

⏳ Memisahkan data Training (2021-2024) dan Testing (2025)...

✔️ Sukses memuat data ke RAM! Waktu eksekusi: 31.62 detik.
📊 Jumlah Data Training : 2,775,604 baris
📊 Jumlah Data Testing  : 2,724,359 baris
✅ Selesai! Komputasi ML selanjutnya berjalan 100% Offline & Kilat!


## 4. BALANCING DATA SAMPLING (RANDOM UNDERSAMPLING)

In [9]:
import pyspark.sql.functions as F

print("🧹 Membersihkan residu 'NULL' bawaan database (Tahap 2)...")

# Kita HANYA menggunakan string di dalam filter, agar Spark tidak sok pintar melakukan auto-cast
valid_labels = ["0", "1", "0.0", "1.0"]

# 1. Usir "\N" dan filter murni sebagai teks
df_train_raw = df_train_raw.filter(F.col("arr_del15_label") != "\\N") \
                           .filter(F.col("arr_del15_label").isNotNull()) \
                           .filter(F.col("arr_del15_label").isin(valid_labels))

df_test_raw = df_test_raw.filter(F.col("arr_del15_label") != "\\N") \
                         .filter(F.col("arr_del15_label").isNotNull()) \
                         .filter(F.col("arr_del15_label").isin(valid_labels))

# 2. Setelah 100% dijamin hanya berisi "0" atau "1", baru kita paksa jadi Integer
df_train_raw = df_train_raw.withColumn("arr_del15_label", F.col("arr_del15_label").cast("int"))
df_test_raw = df_test_raw.withColumn("arr_del15_label", F.col("arr_del15_label").cast("int"))

# 3. Hitung hasilnya
print(f"📊 Sisa Data Training Bersih : {df_train_raw.count():,} baris")
print(f"📊 Sisa Data Testing Bersih  : {df_test_raw.count():,} baris")
print("✅ Pembersihan selesai total! Silakan lanjut jalankan ulang Cell Balancing.")

🧹 Membersihkan residu 'NULL' bawaan database (Tahap 2)...
📊 Sisa Data Training Bersih : 2,721,602 baris
📊 Sisa Data Testing Bersih  : 2,678,573 baris
✅ Pembersihan selesai total! Silakan lanjut jalankan ulang Cell Balancing.


In [10]:
# print("⚖️ Menjalankan proses balancing (Undersampling) terdistribusi pada data training...")

# # Pisahkan kelas target
# df_ontime = df_train_raw.filter(df_train_raw.arr_del15_label == 0)
# df_delay = df_train_raw.filter(df_train_raw.arr_del15_label == 1)

# count_ontime = df_ontime.count()
# count_delay = df_delay.count()

# # Hitung rasio fraksi untuk mengambil sampel acak data mayoritas (ontime)
# fraction = count_delay / count_ontime

# # Jalankan undersampling tanpa pengembalian (withReplacement=False)
# df_ontime_sampled = df_ontime.sample(withReplacement=False, fraction=fraction, seed=42)

# # Satukan kembali data training yang rasionya sudah seimbang 50:50
# df_train_balanced = df_delay.union(df_ontime_sampled)

# print(f"✅ Balancing selesai!")
# print(f"📈 Total Data Training Setelah Diet Rasio: {df_train_balanced.count():,} baris.")

In [11]:
# ============================================================
# GANTI SELURUH BLOK UNDERSAMPLING DENGAN INI
# ============================================================

# Hitung bobot berdasarkan distribusi ASLI training data
count_total = df_train_raw.count()
count_delay = df_train_raw.filter(F.col("arr_del15_label") == 1).count()
count_ontime = df_train_raw.filter(F.col("arr_del15_label") == 0).count()

# Bobot berbanding terbalik dengan frekuensi kelas
weight_delay  = count_total / (2.0 * count_delay)   # ~2.32
weight_ontime = count_total / (2.0 * count_ontime)  # ~0.80

df_train_raw = df_train_raw.withColumn(
    "class_weight",
    F.when(F.col("arr_del15_label") == 1, weight_delay)
     .otherwise(weight_ontime)
)

# Tambahkan time_weight untuk downweight era pandemi
df_train_raw = df_train_raw.withColumn(
    "time_weight",
    F.when(F.col("FlightDate") < "2023-01-01", 0.5)
     .otherwise(1.0)
)

# Gabungkan jadi final_weight
df_train_raw = df_train_raw.withColumn(
    "final_weight",
    F.col("class_weight") * F.col("time_weight")
)

df_train_balanced = df_train_raw

In [12]:
# Cek apakah time_weight benar-benar bervariasi
print("Distribusi time_weight:")
df_train_raw.groupBy("time_weight").count().show()

print("Distribusi final_weight (sample):")
df_train_raw.select("time_weight", "class_weight", "final_weight") \
    .groupBy("time_weight", "class_weight") \
    .count().show()

Distribusi time_weight:
+-----------+-------+
|time_weight|  count|
+-----------+-------+
|        1.0|2721602|
+-----------+-------+

Distribusi final_weight (sample):
+-----------+------------------+-------+
|time_weight|      class_weight|  count|
+-----------+------------------+-------+
|        1.0|0.6369079657432881|2136574|
|        1.0| 2.326044223524344| 585028|
+-----------+------------------+-------+



## 5. FEATURE ENGINEERING & PREPROCESSING

In [13]:
df_train_balanced.select("IATA_CODE_Reporting_Airline").distinct().count()

15

In [14]:
print("⚙️ Memulai transformasi fitur numerik dan kategorikal...")

# ── StringIndexer untuk kategorikal ──────────────────────────────
indexer_season   = StringIndexer(inputCol="season",    outputCol="season_index",   handleInvalid="keep")
indexer_dist     = StringIndexer(inputCol="distance_bucket", outputCol="distance_index", handleInvalid="keep")
indexer_airline  = StringIndexer(inputCol="IATA_CODE_Reporting_Airline", outputCol="airline_index", handleInvalid="keep")

fit_season   = indexer_season.fit(df_train_balanced)
fit_dist     = indexer_dist.fit(df_train_balanced)
fit_airline  = indexer_airline.fit(df_train_balanced)

def apply_indexers(df):
    df = fit_season.transform(df)
    df = fit_dist.transform(df)
    df = fit_airline.transform(df)
    return df

df_train_indexed = apply_indexers(df_train_balanced)
df_test_indexed  = apply_indexers(df_test_raw)

# ── Cast kolom string → double ────────────────────────────────────
kolom_numerik = [
    "route_avg_arr_delay_prev",
    "carrier_arr_delay_rate_prev",
    "flight_month", "dep_hour", "arr_hour",
    "is_weekend", "same_state_route",
    "day_of_week", "CRSElapsedTime", "Distance", "flight_day",
    "route_arr_delay_rate_prev",
    "route_carrier_arr_delay_rate_prev",
    "origin_arr_delay_rate_prev", "origin_cancel_rate_prev",
    "dest_arr_delay_rate_prev",   "dest_cancel_rate_prev",
    "carrier_cancel_rate_prev",
]
for col_name in kolom_numerik:
    df_train_indexed = df_train_indexed.withColumn(col_name, F.col(col_name).cast("double"))
    df_test_indexed  = df_test_indexed.withColumn(col_name,  F.col(col_name).cast("double"))

# ── Flag has_route_history SEBELUM fillna ─────────────────────────
df_train_indexed = df_train_indexed.withColumn(
    "has_route_history",
    F.when(F.col("route_avg_arr_delay_prev").isNull(), 0.0).otherwise(1.0)
)
df_test_indexed = df_test_indexed.withColumn(
    "has_route_history",
    F.when(F.col("route_avg_arr_delay_prev").isNull(), 0.0).otherwise(1.0)
)

# ── Hitung median dari df_train_raw untuk imputasi ───────────────
df_train_raw_cast = df_train_raw
for col_name in kolom_numerik:
    df_train_raw_cast = df_train_raw_cast.withColumn(col_name, F.col(col_name).cast("double"))

kolom_untuk_median = [
    "route_avg_arr_delay_prev", "carrier_arr_delay_rate_prev",
    "route_arr_delay_rate_prev", "route_carrier_arr_delay_rate_prev",
    "origin_arr_delay_rate_prev", "origin_cancel_rate_prev",
    "dest_arr_delay_rate_prev", "dest_cancel_rate_prev",
    "carrier_cancel_rate_prev",
]
medians = {}
for col_name in kolom_untuk_median:
    medians[col_name] = df_train_raw_cast.approxQuantile(col_name, [0.5], 0.01)[0]
    print(f"  Median {col_name}: {medians[col_name]:.4f}")

# Terapkan fillna median untuk kolom historis
for col_name in kolom_untuk_median:
    df_train_indexed = df_train_indexed.fillna(medians[col_name], subset=[col_name])
    df_test_indexed  = df_test_indexed.fillna(medians[col_name],  subset=[col_name])

# fillna 0.0 untuk kolom non-historis
other_cols = ["flight_month", "dep_hour", "arr_hour", "is_weekend",
              "same_state_route", "day_of_week", "CRSElapsedTime",
              "Distance", "flight_day", "OriginAirportID", "DestAirportID"]
df_train_indexed = df_train_indexed.fillna(0.0, subset=other_cols)
df_test_indexed  = df_test_indexed.fillna(0.0,  subset=other_cols)

print("✅ Imputasi selesai!")

# ── Encoding siklikal ─────────────────────────────────────────────
for df_name, df in [("train", df_train_indexed), ("test", df_test_indexed)]:
    df = df \
        .withColumn("dep_hour_sin",  F.sin(2 * 3.14159 * F.col("dep_hour") / 24)) \
        .withColumn("dep_hour_cos",  F.cos(2 * 3.14159 * F.col("dep_hour") / 24)) \
        .withColumn("arr_hour_sin",  F.sin(2 * 3.14159 * F.col("arr_hour") / 24)) \
        .withColumn("arr_hour_cos",  F.cos(2 * 3.14159 * F.col("arr_hour") / 24)) \
        .withColumn("month_sin",     F.sin(2 * 3.14159 * F.col("flight_month") / 12)) \
        .withColumn("month_cos",     F.cos(2 * 3.14159 * F.col("flight_month") / 12)) \
        .withColumn("dow_sin",       F.sin(2 * 3.14159 * F.col("day_of_week") / 7)) \
        .withColumn("dow_cos",       F.cos(2 * 3.14159 * F.col("day_of_week") / 7)) \
        .withColumn("carrier_rate_x_peak_hour",
                    F.col("carrier_arr_delay_rate_prev") * F.col("dep_hour_sin"))
    if df_name == "train":
        df_train_indexed = df
    else:
        df_test_indexed = df

# ── FEATURE_COLS final ────────────────────────────────────────────
FEATURE_COLS = [
    # Temporal (siklikal)
    "dep_hour_sin", "dep_hour_cos",
    "arr_hour_sin", "arr_hour_cos",
    "month_sin", "month_cos",
    "dow_sin", "dow_cos",
    "is_weekend", "flight_day",

    # Rute & maskapai (encoded)
    "airline_index", "OriginAirportID", "DestAirportID",
    "season_index", "same_state_route",

    # Numerik operasional
    "Distance", "CRSElapsedTime",

    # Historis — INI YANG PALING PENTING
    "route_avg_arr_delay_prev",
    "route_arr_delay_rate_prev",
    "route_carrier_arr_delay_rate_prev",
    "carrier_arr_delay_rate_prev",
    "carrier_cancel_rate_prev",
    "origin_arr_delay_rate_prev",
    "origin_cancel_rate_prev",
    "dest_arr_delay_rate_prev",
    "dest_cancel_rate_prev",
    "has_route_history",

    # Interaksi
    "carrier_rate_x_peak_hour",
]

# ── VectorAssembler ───────────────────────────────────────────────
assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features", handleInvalid="keep")

df_train_final = assembler.transform(df_train_indexed).select("features", "arr_del15_label", "final_weight")
df_test_final  = assembler.transform(df_test_indexed).select("features", "arr_del15_label")

print(f"✅ Pipeline siap! Total fitur: {len(FEATURE_COLS)}")

⚙️ Memulai transformasi fitur numerik dan kategorikal...
  Median route_avg_arr_delay_prev: 16.2128
  Median carrier_arr_delay_rate_prev: 0.2077
  Median route_arr_delay_rate_prev: 0.2143
  Median route_carrier_arr_delay_rate_prev: 0.2100
  Median origin_arr_delay_rate_prev: 0.2150
  Median origin_cancel_rate_prev: 0.0138
  Median dest_arr_delay_rate_prev: 0.2140
  Median dest_cancel_rate_prev: 0.0130
  Median carrier_cancel_rate_prev: 0.0129
✅ Imputasi selesai!
✅ Pipeline siap! Total fitur: 28


## 5.1. DIAGNOSTIK DATA

In [15]:
print("🔍 Memeriksa distribusi label dan kualitas fitur sebelum training...")

print("\n📌 Ukuran dataset final:")
print(f"  Training rows: {df_train_final.count():,}")
print(f"  Testing rows : {df_test_final.count():,}")

print("\n📊 Distribusi label Training:")
df_train_balanced.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

print("\n📊 Distribusi label Testing:")
df_test_raw.groupBy("arr_del15_label").count().orderBy("arr_del15_label").show()

# Baseline accuracy dengan majority class pada data test
positive_test = df_test_raw.filter(F.col("arr_del15_label") == 1).count()
negative_test = df_test_raw.filter(F.col("arr_del15_label") == 0).count()
if positive_test + negative_test > 0:
    positive_rate = positive_test / (positive_test + negative_test)
    baseline_accuracy = max(positive_rate, 1.0 - positive_rate)
    print(f"\n🔢 Positive rate (delay) on test set: {positive_rate:.4f}")
    print(f"🔢 Baseline accuracy (majority class): {baseline_accuracy:.4f}")
else:
    print("⚠️ Tidak ada data test untuk menghitung baseline")

print("\n🧪 Missing / null count on key features in training set:")
missing_cols = [
    'route_avg_arr_delay_prev',
    'carrier_arr_delay_rate_prev',
    'flight_month',
    'dep_hour',
    'is_weekend',
    'same_state_route',
    'season_index',
    'distance_index'
]
missing_exprs = [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in missing_cols]
missing_counts = df_train_indexed.select(missing_exprs)
missing_counts.show()

print("\n📌 Distinct categories for encoded categorical features:")
df_train_indexed.select("season_index", "distance_index").distinct().show(50)

🔍 Memeriksa distribusi label dan kualitas fitur sebelum training...

📌 Ukuran dataset final:
  Training rows: 2,721,602
  Testing rows : 2,678,573

📊 Distribusi label Training:
+---------------+-------+
|arr_del15_label|  count|
+---------------+-------+
|              0|2136574|
|              1| 585028|
+---------------+-------+


📊 Distribusi label Testing:
+---------------+-------+
|arr_del15_label|  count|
+---------------+-------+
|              0|2079771|
|              1| 598802|
+---------------+-------+


🔢 Positive rate (delay) on test set: 0.2236
🔢 Baseline accuracy (majority class): 0.7764

🧪 Missing / null count on key features in training set:
+------------------------+---------------------------+------------+--------+----------+----------------+------------+--------------+
|route_avg_arr_delay_prev|carrier_arr_delay_rate_prev|flight_month|dep_hour|is_weekend|same_state_route|season_index|distance_index|
+------------------------+---------------------------+------------+

## 6. TRAINING MODEL

In [16]:
print("🤖 Mempersiapkan inisialisasi algoritma target...")

# Definisikan model non-XGBoost terlebih dahulu
spark_models = {
    "Logistic Regression": LogisticRegression(featuresCol="features", labelCol="arr_del15_label", weightCol="final_weight", maxIter=100),
    "Random Forest": RandomForestClassifier(featuresCol="features", labelCol="arr_del15_label", weightCol="final_weight", numTrees=100, seed=42)
}

xgboost_model = SparkXGBClassifier(features_col="features", label_col="arr_del15_label", weight_col="final_weight", num_workers=1, eval_metric="logloss")

# Wadah untuk menyimpan model yang sudah terlatih
trained_spark_models = {}

print("\n🚀 Memulai proses training non-XGBoost di cluster Spark...")
for name, model in spark_models.items():
    print(f"   ⏳ Sedang melatih model: {name}...")
    trained_spark_models[name] = model.fit(df_train_final)
    print(f"   ✔️ {name} selesai dilatih!")

print("\n⚠️ Menjalankan XGBoost secara terpisah agar kegagalan worker tidak memutus pipeline...")
try:
    trained_spark_models["XGBoost"] = xgboost_model.fit(df_train_final)
    print("   ✔️ XGBoost selesai dilatih!")
except Exception as e:
    print("   ❌ XGBoost gagal. Spark Python worker mungkin tidak terhubung dengan benar.")
    print(str(e))
    trained_spark_models["XGBoost"] = None

print("\n🎉 Model non-XGBoost selesai dilatih. XGBoost diisolasi untuk debugging.")

🤖 Mempersiapkan inisialisasi algoritma target...

🚀 Memulai proses training non-XGBoost di cluster Spark...
   ⏳ Sedang melatih model: Logistic Regression...
   ✔️ Logistic Regression selesai dilatih!
   ⏳ Sedang melatih model: Random Forest...
   ✔️ Random Forest selesai dilatih!

⚠️ Menjalankan XGBoost secara terpisah agar kegagalan worker tidak memutus pipeline...


INFO:XGBoost-PySpark:Running xgboost-3.3.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'eval_metric': 'logloss', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


   ✔️ XGBoost selesai dilatih!

🎉 Model non-XGBoost selesai dilatih. XGBoost diisolasi untuk debugging.


## 6.1. MODEL INSPEKSI

In [17]:
print("🔎 Memeriksa model dan feature importance...")

feature_names = FEATURE_COLS

log_reg = trained_spark_models.get("Logistic Regression")
if log_reg is not None:
    print("\n📈 Logistic Regression coefficients:")
    coeffs = log_reg.coefficients.toArray() if hasattr(log_reg.coefficients, "toArray") else list(log_reg.coefficients)
    for name, coef in zip(feature_names, coeffs):
        print(f"  {name}: {coef:.6f}")

rf_model = trained_spark_models.get("Random Forest")
if rf_model is not None:
    print("\n🌲 Random Forest feature importances:")
    importances = rf_model.featureImportances.toArray() if hasattr(rf_model.featureImportances, "toArray") else list(rf_model.featureImportances)
    sorted_features = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)
    for name, imp in sorted_features:
        print(f"  {name}: {imp:.6f}")

🔎 Memeriksa model dan feature importance...

📈 Logistic Regression coefficients:
  dep_hour_sin: 0.036517
  dep_hour_cos: -0.015720
  arr_hour_sin: -0.194865
  arr_hour_cos: 0.137939
  month_sin: -0.206061
  month_cos: -0.346318
  dow_sin: -0.123448
  dow_cos: 0.112467
  is_weekend: -0.140059
  flight_day: 0.012628
  airline_index: 0.002877
  OriginAirportID: 0.000029
  DestAirportID: -0.000000
  season_index: -0.346597
  same_state_route: 0.043927
  Distance: 0.000235
  CRSElapsedTime: -0.002033
  route_avg_arr_delay_prev: 0.001354
  route_arr_delay_rate_prev: 0.369982
  route_carrier_arr_delay_rate_prev: 4.468675
  carrier_arr_delay_rate_prev: 1.833388
  carrier_cancel_rate_prev: -2.783714
  origin_arr_delay_rate_prev: -0.113184
  origin_cancel_rate_prev: 0.518434
  dest_arr_delay_rate_prev: 0.393411
  dest_cancel_rate_prev: 2.960040
  has_route_history: 0.000000
  carrier_rate_x_peak_hour: -1.529162

🌲 Random Forest feature importances:
  carrier_rate_x_peak_hour: 0.206919
  dep_hou

## 7. EVALUASI MODEL

In [18]:
print("🕵️ Memulai proses evaluasi massal pada Data Testing (Data 2025)...")

# Gunakan rawPrediction (bukan probability) untuk ROC-AUC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="arr_del15_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

evaluator_pr = BinaryClassificationEvaluator(
    labelCol="arr_del15_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"   # tambahan: PR-AUC lebih informatif untuk kelas imbalanced
)

final_eval_rows = []
all_predictions = {}  # simpan untuk threshold tuning di bawah

for name, model in trained_spark_models.items():
    if model is None:
        print(f"\n⚠️ Skipping {name} karena model tidak tersedia.")
        continue

    predictions = model.transform(df_test_final).cache()
    all_predictions[name] = predictions  # simpan untuk threshold tuning

    roc_auc = evaluator_auc.evaluate(predictions)
    pr_auc  = evaluator_pr.evaluate(predictions)

    pred_labels = predictions.select("prediction", "arr_del15_label")
    tp = pred_labels.filter((F.col("prediction") == 1.0) & (F.col("arr_del15_label") == 1)).count()
    tn = pred_labels.filter((F.col("prediction") == 0.0) & (F.col("arr_del15_label") == 0)).count()
    fp = pred_labels.filter((F.col("prediction") == 1.0) & (F.col("arr_del15_label") == 0)).count()
    fn = pred_labels.filter((F.col("prediction") == 0.0) & (F.col("arr_del15_label") == 1)).count()

    total   = tp + tn + fp + fn
    accuracy  = (tp + tn) / total if total > 0 else 0.0
    precision = tp / (tp + fp)    if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn)    if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0  # tambahan: True Negative Rate

    print(f"\n🧾 [{name}]")
    print(f"   Confusion Matrix → TP={tp:,}  TN={tn:,}  FP={fp:,}  FN={fn:,}")
    print(f"   Accuracy={accuracy:.4f} | Precision={precision:.4f} | Recall={recall:.4f}")
    print(f"   F1={f1:.4f} | ROC-AUC={roc_auc:.4f} | PR-AUC={pr_auc:.4f} | Specificity={specificity:.4f}")

    final_eval_rows.append({
        "Algoritma":   name,
        "Accuracy":    f"{accuracy:.4f}",
        "Precision":   f"{precision:.4f}",
        "Recall":      f"{recall:.4f}",
        "Specificity": f"{specificity:.4f}",
        "F1-Score":    f"{f1:.4f}",
        "ROC-AUC":     f"{roc_auc:.4f}",
        "PR-AUC":      f"{pr_auc:.4f}",
    })

print("\n📊 TABEL EVALUASI AKHIR PIPELINE APACHE SPARK (3 MODEL):")
print("=" * 85)
df_spark_eval = pd.DataFrame(final_eval_rows)
print(df_spark_eval.to_string(index=False))
print("=" * 85)

# Baseline: jika model selalu prediksi majority class (on-time)
baseline_acc = 2079771 / (2079771 + 598802)
print(f"\n📌 Baseline majority-class accuracy: {baseline_acc:.4f}")
print(f"   (Model yang bagus harus jauh di atas ini)")

🕵️ Memulai proses evaluasi massal pada Data Testing (Data 2025)...

🧾 [Logistic Regression]
   Confusion Matrix → TP=354,488  TN=1,295,340  FP=784,431  FN=244,314
   Accuracy=0.6159 | Precision=0.3112 | Recall=0.5920
   F1=0.4080 | ROC-AUC=0.6460 | PR-AUC=0.3364 | Specificity=0.6228

🧾 [Random Forest]
   Confusion Matrix → TP=371,403  TN=1,251,959  FP=827,812  FN=227,399
   Accuracy=0.6061 | Precision=0.3097 | Recall=0.6202
   F1=0.4131 | ROC-AUC=0.6526 | PR-AUC=0.3396 | Specificity=0.6020

🧾 [XGBoost]
   Confusion Matrix → TP=329,078  TN=1,366,031  FP=713,740  FN=269,724
   Accuracy=0.6328 | Precision=0.3156 | Recall=0.5496
   F1=0.4009 | ROC-AUC=0.6464 | PR-AUC=0.3352 | Specificity=0.6568

📊 TABEL EVALUASI AKHIR PIPELINE APACHE SPARK (3 MODEL):
          Algoritma Accuracy Precision Recall Specificity F1-Score ROC-AUC PR-AUC
Logistic Regression   0.6159    0.3112 0.5920      0.6228   0.4080  0.6460 0.3364
      Random Forest   0.6061    0.3097 0.6202      0.6020   0.4131  0.6526 0.33

In [19]:
print("🎯 Mencari threshold optimal untuk setiap model (efisien via pandas)...")
extract_prob = F.udf(lambda v: float(v[1]), "double")

for name, predictions in all_predictions.items():
    print(f"\n🔍 Model: {name}")

    # Tarik ke pandas SEKALI — jauh lebih cepat dari 40x Spark jobs
    df_local = predictions.select(
        "arr_del15_label",
        extract_prob(F.col("probability")).alias("prob_delay")
    ).toPandas()

    y_true  = df_local["arr_del15_label"].values
    y_proba = df_local["prob_delay"].values

    best_f1, best_thresh, best_prec, best_rec = 0, 0.5, 0, 0

    for thresh in np.arange(0.20, 0.80, 0.01):
        y_pred = (y_proba >= thresh).astype(int)
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        fn = ((y_pred == 0) & (y_true == 1)).sum()

        prec = tp / (tp + fp + 1e-9)
        rec  = tp / (tp + fn + 1e-9)
        f1   = 2 * prec * rec / (prec + rec + 1e-9)

        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
            best_prec, best_rec  = prec, rec

    print(f"   Best threshold : {best_thresh:.2f}")
    print(f"   F1={best_f1:.4f} | Precision={best_prec:.4f} | Recall={best_rec:.4f}")
    print(f"   → Default threshold 0.5 vs optimal {best_thresh:.2f}: "
          f"{'naik' if best_f1 > 0 else 'sama'}")

🎯 Mencari threshold optimal untuk setiap model (efisien via pandas)...

🔍 Model: Logistic Regression
   Best threshold : 0.47
   F1=0.4097 | Precision=0.2979 | Recall=0.6555
   → Default threshold 0.5 vs optimal 0.47: naik

🔍 Model: Random Forest
   Best threshold : 0.48
   F1=0.4145 | Precision=0.2999 | Recall=0.6705
   → Default threshold 0.5 vs optimal 0.48: naik

🔍 Model: XGBoost
   Best threshold : 0.43
   F1=0.4095 | Precision=0.2921 | Recall=0.6846
   → Default threshold 0.5 vs optimal 0.43: naik


## 8. MENYIMPAN ARTIFACT MODEL

In [24]:
os.makedirs("models", exist_ok=True)

for name, model in trained_spark_models.items():
    if model is None:
        print(f"⚠️ Skip {name} — model tidak tersedia")
        continue

    # Buat path folder untuk model
    save_path = os.path.join("..", "models", name.lower().replace(' ', '_'))

    # Hapus folder lama jika sudah ada (Spark tidak bisa overwrite)
    import shutil
    if os.path.exists(save_path):
        shutil.rmtree(save_path)

    model.save(save_path)
    print(f"✔️ Tersimpan: {save_path}/")

print("\n✅ Semua model berhasil disimpan!")

# # Untuk load kembali lagi nanti:
# from pyspark.ml.classification import LogisticRegressionModel, RandomForestClassificationModel
# from xgboost.spark import SparkXGBClassifierModel

# lr_model  = LogisticRegressionModel.load("models/logistic_regression")
# rf_model  = RandomForestClassificationModel.load("models/random_forest")
# xgb_model = SparkXGBClassifierModel.load("models/xgboost")  # jika berhasil dilatih

✔️ Tersimpan: ..\models\logistic_regression/
✔️ Tersimpan: ..\models\random_forest/
✔️ Tersimpan: ..\models\xgboost/

✅ Semua model berhasil disimpan!


In [25]:
import os, shutil

os.makedirs("../models", exist_ok=True)

# Simpan encoders
encoders = {
    "encoder_season":  fit_season,
    "encoder_dist":    fit_dist,
    "encoder_airline": fit_airline,
}

for name, encoder in encoders.items():
    save_path = f"../models/{name}"
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    encoder.save(save_path)
    print(f"✔️ Tersimpan: {save_path}/")

print("\n✅ Semua encoder berhasil disimpan!")


# # Untuk load kembali nanti:
# from pyspark.ml.feature import StringIndexerModel

# fit_season  = StringIndexerModel.load("../models/encoder_season")
# fit_dist    = StringIndexerModel.load("../models/encoder_dist")
# fit_airline = StringIndexerModel.load("../models/encoder_airline")

✔️ Tersimpan: ../models/encoder_season/
✔️ Tersimpan: ../models/encoder_dist/
✔️ Tersimpan: ../models/encoder_airline/

✅ Semua encoder berhasil disimpan!
